<h3>Build an ANN by implementing of backpropogation algorithm and test the same using a apporpiate dataset</h3>

In [65]:
# import liberaries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [66]:
df = pd.read_csv("Mnist/train.csv") # Readdatset
df.sample(3)

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
970,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
28374,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
764,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [82]:
# Data preproessing
data = np.array(df)
m, n = data.shape
np.random.shuffle(data) 

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]
X_dev = X_dev / 255.

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n]
X_train = X_train / 255.

In [84]:
# init neural network
def init_params():
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

# implementation of activation functions
def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

# Forward propagation     
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def ReLU_deriv(Z):
    return Z > 0

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, int(Y.max()) + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

# Categorical Cross-Entropy Loss computation
def get_loss(A2, Y):
    one_hot_Y = one_hot(Y)
    eps = 1e-15
    A2_clipped = np.clip(A2, eps, 1 - eps)
    return -np.sum(one_hot_Y * np.log(A2_clipped)) / Y.size

# Backpropagation
def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    m_samples = X.shape[1]
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m_samples * dZ2.dot(A1.T)
    db2 = 1 / m_samples * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m_samples * dZ1.dot(X.T)
    db1 = 1 / m_samples * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

# Updating parameters
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2


In [88]:
import copy

def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size

# Gradient Descent with EarlyStopping & ModelCheckpoint
def gradient_descent(X, Y, alpha, iterations, X_val=None, Y_val=None, patience=20, eval_freq=10, checkpoint_path='best_ann_weights.npz'):
    W1, b1, W2, b2 = init_params()
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    eval_iters = []
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_weights = None
    
    for i in range(1, iterations + 1):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        
        if i % eval_freq == 0 or i == iterations:
            train_loss = get_loss(A2, Y)
            train_acc = get_accuracy(get_predictions(A2), Y)
            
            if X_val is not None and Y_val is not None:
                _, _, _, A2_val = forward_prop(W1, b1, W2, b2, X_val)
                val_loss = get_loss(A2_val, Y_val)
                val_acc = get_accuracy(get_predictions(A2_val), Y_val)
            else:
                val_loss, val_acc = train_loss, train_acc
                
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_accs.append(train_acc)
            val_accs.append(val_acc)
            eval_iters.append(i)
            
            print(f'Iteration {i}: Train Loss = {round(train_loss, 4)}, Train Acc = {round(train_acc, 4)} | Val Loss = {round(val_loss, 4)}, Val Acc = {round(val_acc, 4)}')
            
            # ModelCheckpoint: Save exclusively the absolute best epoch weights payload locally
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_weights = (copy.deepcopy(W1), copy.deepcopy(b1), copy.deepcopy(W2), copy.deepcopy(b2))
                np.savez(checkpoint_path, W1=W1, b1=b1, W2=W2, b2=b2)
                print(f'--> Checkpoint saved at iteration {i} with Val Loss {round(best_val_loss, 4)}')
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f'EarlyStopping triggered at iteration {i}! Restoring best model weights.')
                    W1, b1, W2, b2 = best_weights
                    break
                    
    history = {
        'iters': eval_iters,
        'train_loss': train_losses,
        'val_loss': val_losses,
        'train_acc': train_accs,
        'val_acc': val_accs
    }
    return W1, b1, W2, b2, history


In [89]:
# Model training with early stopping & model checkpointing on validation set
W1, b1, W2, b2, history = gradient_descent(X_train, Y_train, 0.10, 1000, X_val=X_dev, Y_val=Y_dev, patience=20, eval_freq=10)


[5 8 8 ... 0 7 7] [8 8 8 ... 0 7 7]
0.8493658536585366


In [ ]:
# Plotting validation curves alongside training curve comparisons
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['iters'], history['train_loss'], label='Training Loss')
plt.plot(history['iters'], history['val_loss'], label='Validation Loss', color='red', linestyle='--')
plt.title('ANN Loss Convergence Curve')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['iters'], history['train_acc'], label='Training Accuracy')
plt.plot(history['iters'], history['val_acc'], label='Validation Accuracy', color='green', linestyle='--')
plt.title('ANN Accuracy Curve')
plt.xlabel('Iterations')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
